# Link Corridor Radius Sensitivity

The corridor radius $r = 100 \text{m}$ is a modeling choice based on what a cyclist experiences along a link. We here recompute the link land use shares, components, and main-specification score for other radii. We also check how much the link ranking changes. The goal is to see whether the results depend on the choice of $r$.

Note that running this notebook requires that `pixi run link-score` has been run once for each of these settings in [src/splendor/config.py](../../src/splendor/config.py):
- `CORRIDOR_RADIUS = 50`
- `CORRIDOR_RADIUS = 100` (the default)
- `CORRIDOR_RADIUS = 200`

<br><br>

## 0. Setup

### 0.1. Imports and Config

In [1]:
import pandas as pd

from splendor.config import CORRIDOR_RADIUS, ROAD_STRIP_RADIUS, STUDY_AREA
from splendor.landuse.shares import SHARE_PREFIX, get_share_type_id
from splendor.landuse.types import SHORT_NAMES
from splendor.paths import landuse_path, components_path, splendor_path

LEVEL = "links"
ID_COLUMN = "edge_id"
RADII = (50, 100, 200)
REFERENCE_RADIUS = CORRIDOR_RADIUS
TOP_N = 20

assert REFERENCE_RADIUS in RADII, "CORRIDOR_RADIUS in config is not among the compared radii"

### 0.2. Data

In [2]:
def load_by_radius(path_function) -> dict[int, pd.DataFrame]:
    return {
        r: pd.read_csv(path_function(STUDY_AREA, LEVEL, r, ROAD_STRIP_RADIUS), index_col=ID_COLUMN)
        for r in RADII
    }

landuse = load_by_radius(landuse_path)
components = load_by_radius(components_path)
splendor = load_by_radius(splendor_path)

for frames in (landuse, components, splendor):
    ids = frames[REFERENCE_RADIUS].index
    assert all(df.index.equals(ids) for df in frames.values()), "edge ids differ between radii"

print(f"{len(ids)} links, radii {RADII}")

638 links, radii (50, 100, 200)


<br><br>

## 1. Land Use Shares per Radius

In [3]:
share_columns = [c for c in landuse[REFERENCE_RADIUS].columns if c.startswith(SHARE_PREFIX)]

# Mean share per type over links
mean_shares = pd.DataFrame({r: df[share_columns].mean() for r, df in landuse.items()})
mean_shares.index = [SHORT_NAMES[get_share_type_id(c)] for c in share_columns]
print(mean_shares.sort_values(REFERENCE_RADIUS, ascending=False).round(3))

# Corridor size
print(pd.DataFrame({r: df["n_pixels"].describe() for r, df in landuse.items()}).round(0))

               50     100    200
Cropland     0.557  0.599  0.643
Settlement   0.228  0.191  0.147
Forest       0.086  0.084  0.083
Road/rail    0.077  0.068  0.061
Grassland    0.020  0.021  0.021
Sea          0.008  0.012  0.016
Lakes        0.006  0.007  0.010
Coast        0.008  0.006  0.005
Rivers       0.006  0.006  0.006
Mires        0.004  0.005  0.007
Lagoons      0.000  0.001  0.001
Heathland    0.000  0.000  0.000
Sparse veg.  0.000  0.000  0.000
           50       100      200
count    638.0    638.0    638.0
mean    1926.0   4133.0   8530.0
std     1450.0   3101.0   6385.0
min        7.0     12.0     27.0
25%      978.0   2102.0   4390.0
50%     1709.0   3676.0   7588.0
75%     2596.0   5542.0  11443.0
max    16698.0  35781.0  73909.0


<br><br>

## 2. Components per Radius

In [4]:
component_columns = list(components[REFERENCE_RADIUS].columns)

summary = pd.concat(
    {r: df[component_columns].describe().T[["mean", "std", "min", "50%", "max"]] for r, df in components.items()},
    names=["r", "component"],
)
print(summary.swaplevel().sort_index().round(3))

                         mean    std    min    50%    max
component          r                                     
attractiveness     50   0.216  0.113  0.000  0.212  0.862
                   100  0.229  0.110  0.000  0.224  0.833
                   200  0.246  0.104  0.000  0.234  0.814
diversity_richness 50   2.141  0.719  1.000  2.000  5.000
                   100  2.011  0.773  1.000  2.000  5.000
                   200  1.859  0.857  1.000  2.000  5.000
diversity_shannon  50   0.872  0.322  0.000  0.847  1.913
                   100  0.857  0.337  0.081  0.804  1.931
                   200  0.837  0.361  0.127  0.785  1.900


<br><br>

## 3. Rank Correlations

In [5]:
def get_reference_correlations(frames: dict[int, pd.DataFrame], columns: list[str]) -> pd.DataFrame:
    reference = frames[REFERENCE_RADIUS]
    return pd.DataFrame(
        {r: [df[c].corr(reference[c], method="spearman") for c in columns]
         for r, df in frames.items() if r != REFERENCE_RADIUS},
        index=columns,
    )

correlations = pd.concat([
    get_reference_correlations(components, component_columns),
    get_reference_correlations(splendor, ["score"]),
])
print(correlations.round(3))

# Main score, all pairs
scores = pd.DataFrame({r: df["score"] for r, df in splendor.items()})
print(scores.corr(method="spearman").round(3))

                      50     200
diversity_shannon   0.943  0.924
diversity_richness  0.749  0.733
attractiveness      0.981  0.949
score               0.961  0.950
       50     100    200
50   1.000  0.961  0.874
100  0.961  1.000  0.950
200  0.874  0.950  1.000


<br><br>

## 4. Top and Bottom Links

In [6]:
ranks = pd.DataFrame({r: df["rank"] for r, df in splendor.items()}) # Rank 1 is the best link

def get_extreme_ids(rank: pd.Series, n: int, which: str) -> set:
    return set((rank.nsmallest(n) if which == "top" else rank.nlargest(n)).index)

overlap = pd.DataFrame({
    which: {
        r: len(get_extreme_ids(ranks[r], TOP_N, which) & get_extreme_ids(ranks[REFERENCE_RADIUS], TOP_N, which))
        for r in RADII if r != REFERENCE_RADIUS
    }
    for which in ("top", "bottom")
})
print(f"Links shared with the {TOP_N} at r = {REFERENCE_RADIUS} m:")
print(overlap)

# Where the reference top links rank at the other radii
top_ids = ranks[REFERENCE_RADIUS].nsmallest(TOP_N).index
print(ranks.loc[top_ids].sort_values(REFERENCE_RADIUS).astype(int))

Links shared with the 20 at r = 100 m:
     top  bottom
50    19      15
200   15      12
         50   100  200
edge_id               
174        1    1    1
195        3    2    2
146        4    3    3
589       12    4    6
577        2    5    8
18         5    6    9
63        14    7    5
188       16    8    4
198        6    9   13
254        8   10    7
566        9   11   16
571        7   12   23
463       11   13   11
350       13   14   12
583       10   15   26
529       18   16   10
400       15   17   27
130       42   18   20
138       19   19   28
0         20   20   22


<br><br>

## 5. Conclusion

The link ranking is robust to the corridor radius. The main score score correlates 0.96 with $r = 50$ and 0.95 with $r = 200$, and the correlation between the two extremes is 0.87. Of the top 20 links at $r = 100$, 19 ($r = 50$) and 15 ($r = 200$) stay. Of the bottom 20, 15 and 12 stay. Shannon entropy and attractiveness are also stable (0.92-0.98). Richness is less so (0.73-0.75), but that isn't part of the main specification. 

Larger radii shift the shares from settlemtn and road/rail toward cropland, since settlement concentrates near the road. Diversity therefore decreases slightly and attractiveness increases slightly.

We therefore keep $r = 100 \text{m}$.